# Purpose
Determine whether the raw sensor data is trustworthy enough to analyze.

In [ ]:
%pip install pandas numpy

In [53]:
import numpy as np
import pandas as pd
import time

In [69]:
db = pd.read_csv("../testing/csv_data_saving/csv/sensor_data.csv")
print(db)


               timestamp  gas_raw  temperature_c  humidity_percent
0    2026-09-23 16:27:19     1456           24.0              42.9
1    2026-09-23 16:27:24     1692           24.0              42.9
2    2026-09-23 16:27:29     1360           24.0              42.9
3    2026-09-23 16:27:34     1085           23.5              42.9
4    2026-09-23 16:27:39      913           24.0              42.8
..                   ...      ...            ...               ...
899  2026-09-23 17:42:52      129           24.0              40.5
900  2026-09-23 17:42:52       54           24.0              40.4
901  2026-09-23 17:42:52      129           24.0              40.2
902  2026-09-23 17:42:52      128           24.0              40.0
903  2026-09-23 17:42:56      129           23.5              39.8

[904 rows x 4 columns]


We are currently expecting the following columns: timestamp, gas_raw, temperature_c, humidity_percent, location, session_id

temporarily we will fill location and session_id manually until we have the permanent MQTT data storage system

In [55]:
db["location"] = "House_01"
db["session_id"] = "ESP32_01"

print(db)

               timestamp  gas_raw  temperature_c  humidity_percent  location  \
0    2026-09-23 16:27:19     1456           24.0              42.9  House_01   
1    2026-09-23 16:27:24     1692           24.0              42.9  House_01   
2    2026-09-23 16:27:29     1360           24.0              42.9  House_01   
3    2026-09-23 16:27:34     1085           23.5              42.9  House_01   
4    2026-09-23 16:27:39      913           24.0              42.8  House_01   
..                   ...      ...            ...               ...       ...   
899  2026-09-23 17:42:52      129           24.0              40.5  House_01   
900  2026-09-23 17:42:52       54           24.0              40.4  House_01   
901  2026-09-23 17:42:52      129           24.0              40.2  House_01   
902  2026-09-23 17:42:52      128           24.0              40.0  House_01   
903  2026-09-23 17:42:56      129           23.5              39.8  House_01   

    session_id  
0     ESP32_01  
1    

### Check for missing values

In [56]:
missing_values = db.isna().sum()
empty_values = db.astype("string").apply(lambda column: column.str.strip().eq("").sum())
missing_summary = pd.DataFrame({"missing": missing_values, "empty": empty_values})
print(missing_summary)

if missing_summary.to_numpy().sum() == 0:
    print("No missing or empty values found.")
else:
    print("Missing or empty values found.")

                  missing  empty
timestamp               0      0
gas_raw                 0      0
temperature_c           0      0
humidity_percent        0      0
location                0      0
session_id              0      0
No missing or empty values found.


In [57]:
print(db.describe())

           gas_raw  temperature_c  humidity_percent
count   904.000000     904.000000        904.000000
mean    160.038717      23.823285         40.250774
std     120.073794       0.294058          0.889710
min      37.000000      23.000000         39.100000
25%     118.000000      23.750000         39.700000
50%     128.000000      23.750000         40.000000
75%     144.000000      24.000000         40.300000
max    1692.000000      24.500000         44.400000


### Check sampling interval
Expected: 5s

In [58]:
np_timestamp = db["timestamp"].to_numpy()
db_time = []

for i in np_timestamp:
    n = i.split(" ")
    db_time.append(n[1])

print(db_time)


['16:27:19', '16:27:24', '16:27:29', '16:27:34', '16:27:39', '16:27:44', '16:27:49', '16:27:54', '16:27:59', '16:28:04', '16:28:09', '16:28:14', '16:28:19', '16:28:24', '16:28:29', '16:28:34', '16:28:39', '16:28:44', '16:28:49', '16:28:54', '16:28:59', '16:29:04', '16:29:09', '16:29:14', '16:29:19', '16:29:25', '16:29:30', '16:29:35', '16:29:40', '16:29:45', '16:29:50', '16:29:55', '16:30:00', '16:30:05', '16:30:10', '16:30:15', '16:30:20', '16:30:25', '16:30:30', '16:30:35', '16:30:40', '16:30:45', '16:30:50', '16:30:55', '16:31:00', '16:31:05', '16:31:10', '16:31:15', '16:31:20', '16:31:25', '16:31:30', '16:31:35', '16:31:40', '16:31:45', '16:31:50', '16:31:55', '16:32:00', '16:32:05', '16:32:10', '16:32:15', '16:32:20', '16:32:25', '16:32:30', '16:32:35', '16:32:40', '16:32:46', '16:32:51', '16:32:56', '16:33:01', '16:33:06', '16:33:11', '16:33:16', '16:33:21', '16:33:26', '16:33:31', '16:33:36', '16:33:41', '16:33:46', '16:33:51', '16:33:56', '16:34:01', '16:34:06', '16:34:11', '16

In [59]:
db_time_seconds = []
for i in db_time:
    n = i.split(":")
    db_time_seconds.append((int(n[0]) * 3600) + (int(n[1]) * 60) + int(n[2]))
print(db_time_seconds)
    

[59239, 59244, 59249, 59254, 59259, 59264, 59269, 59274, 59279, 59284, 59289, 59294, 59299, 59304, 59309, 59314, 59319, 59324, 59329, 59334, 59339, 59344, 59349, 59354, 59359, 59365, 59370, 59375, 59380, 59385, 59390, 59395, 59400, 59405, 59410, 59415, 59420, 59425, 59430, 59435, 59440, 59445, 59450, 59455, 59460, 59465, 59470, 59475, 59480, 59485, 59490, 59495, 59500, 59505, 59510, 59515, 59520, 59525, 59530, 59535, 59540, 59545, 59550, 59555, 59560, 59566, 59571, 59576, 59581, 59586, 59591, 59596, 59601, 59606, 59611, 59616, 59621, 59626, 59631, 59636, 59641, 59646, 59651, 59656, 59661, 59666, 59671, 59676, 59681, 59686, 59691, 59696, 59701, 59706, 59711, 59716, 59721, 59726, 59766, 59766, 59767, 59767, 59767, 59767, 59767, 59767, 59772, 59777, 59782, 59787, 59792, 59797, 59802, 59807, 59812, 59817, 59822, 59827, 59832, 59837, 59842, 59847, 59852, 59857, 59862, 59867, 59872, 59877, 59882, 59887, 59892, 59897, 59902, 59907, 59912, 59917, 59922, 59927, 59932, 59937, 59942, 59947, 59952

In [70]:
timestamps = pd.to_datetime(db["timestamp"])
interval_seconds = timestamps.diff().dt.total_seconds()
unexpected_mask = interval_seconds.notna() & interval_seconds.ne(5)

unexpected_rows = pd.DataFrame({
    "previous_index": db.index.to_series().shift(1)[unexpected_mask].astype(int).to_numpy(),
    "current_index": db.index[unexpected_mask],
    "previous_csv_row": (db.index.to_series().shift(1)[unexpected_mask] + 2).astype(int).to_numpy(),
    "current_csv_row": db.index[unexpected_mask] + 2,
    "previous_timestamp": timestamps.shift(1)[unexpected_mask].to_numpy(),
    "current_timestamp": timestamps[unexpected_mask].to_numpy(),
    "interval_seconds": interval_seconds[unexpected_mask].to_numpy(),
})

print(f"Unexpected intervals: {len(unexpected_rows)}")
print(unexpected_rows.to_string(index=False))

Unexpected intervals: 67
 previous_index  current_index  previous_csv_row  current_csv_row  previous_timestamp   current_timestamp  interval_seconds
             24             25                26               27 2026-09-23 16:29:19 2026-09-23 16:29:25               6.0
             64             65                66               67 2026-09-23 16:32:40 2026-09-23 16:32:46               6.0
             97             98                99              100 2026-09-23 16:35:26 2026-09-23 16:36:06              40.0
             98             99               100              101 2026-09-23 16:36:06 2026-09-23 16:36:06               0.0
             99            100               101              102 2026-09-23 16:36:06 2026-09-23 16:36:07               1.0
            100            101               102              103 2026-09-23 16:36:07 2026-09-23 16:36:07               0.0
            101            102               103              104 2026-09-23 16:36:07 2026-09-23 16:36:07  

In [74]:
print(db[901:903])

               timestamp  gas_raw  temperature_c  humidity_percent
901  2026-09-23 17:42:52      129           24.0              40.2
902  2026-09-23 17:42:52      128           24.0              40.0


We have identified anomalous behavior in a subset of samples where two distinct measurements share identical timestamps.

Additionally, we observed significant delays in data collection. We hypothesize this is caused by the current system's reliance on an active host computer; when the machine enters sleep mode, data reading pauses. Transitioning our data ingestion pipeline to MQTT should resolve both issues, as the duplicated timestamps likely occur when the application wakes up and processes queued data from the COM port all at once.